# Module 1.2: Structured Shelf Cabling with UniFi 0.4m Patch Cables & Multi-Gig 2.5GbE Topology

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 1: Hardware Assembly & Physical Cabling

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand the physical and electrical specifications of the **UniFi Patch Cable 10-Pack (`UACC-CABLE-PATCH-0.4M-BL-10`)**—specifically how 34 AWG ultra-thin 0.4m copper patch cords safely carry **10 GbE** and **PoE++ / PoE+++** without thermal degradation.
2. Design and execute a clean, zero-clutter physical shelf layout around the `USW-Enterprise-8-PoE` utilizing the 0.4m (1.3 ft) cable geometry and ±90°/±45° bendable boots.
3. Analyze end-to-end **Multi-Gigabit throughput ceilings** across every link in the 3-node cluster (`homelab-pi` 1 GbE, `homelab-gpu` 1 GbE / optional 2.5 GbE, `UNAS 2` 2.5 GbE, and 5400 RPM HDD sequential I/O).
4. Bind permanent DHCP static IP reservations on your router for **`UNAS 2` (`10.0.0.202`)** and **`USW-Enterprise-8-PoE` (`10.0.0.203`)** and verify physical link speeds across all ports.

## 1.2.1 Conceptual Background: Ultra-Thin Patch Cabling Physics & Multi-Gig Bottleneck Analysis

Messy 6-foot coiled Ethernet cables tangled behind a homelab shelf trap heat, strain RJ45 jacks, and make port troubleshooting painful. Your **10-pack of 0.4 m (1.3 ft) Blue UniFi Patch Cables (`UACC-CABLE-PATCH-0.4M-BL-10`)** is engineered specifically for dense rack and shelf interconnects.

### 1. Engineering Specs & PoE Thermal Physics of `UACC-CABLE-PATCH-0.4M-BL-10`

| Specification | Value | Why It Matters on Your Homelab Shelf |
| :--- | :--- | :--- |
| **SKU & Pack Size** | `UACC-CABLE-PATCH-0.4M-BL-10` (10× Blue) | Uniform color-coding for all homelab core interconnects—5 cables used now, 5 spares ready for future expansion! |
| **Length** | **0.4 m (approx. 1.3 ft / 15.7 in)** | Zero slack loops when your `USW-Enterprise-8-PoE`, `homelab-pi`, `homelab-gpu`, and `UNAS 2` sit neatly clustered on one shelf. |
| **Jacket & Wire Gauge** | **2.5 mm OD** TPE jacket, **34 AWG** stranded copper UTP | Ultra-flexible routing with a tight **21 mm (0.83")** minimum bend radius—never blocks airflow vents on your Zephyrus or fanless switch. |
| **Bendable RJ45 Boots** | **±90° vertical / ±45° horizontal** | Lets cables hug the back of your shelf cleanly without placing lever torque on the switch or Pi's Ethernet jacks. |
| **Rated Speed & Power** | **Up to 10 GbE** (1/2.5/5/10G) & **PoE+++** | Certified to TIA/EIA-568-B.2 & ISO/IEC 11801 for short-run high-speed and high-wattage PoE patch links. |

- **Electrical Engineering Check — Why 34 AWG Is Safe at `0.4 m` for 60W PoE++:**
  Thin 34 AWG copper has higher DC resistance per meter ($R \approx 0.856\ \Omega/\text{m}$ per conductor) than thick 23 AWG Cat 6 riser cable. Over a *30-meter* wall run, 34 AWG would experience severe $I^2 R$ heating and voltage drop! However, at **0.4 m** using 4-pair PoE++ (`802.3bt`, splitting ~1.1A across 4 parallel twisted pairs $\approx 0.275\text{ A}$ per conductor), total conductor resistance per path is tiny ($\sim 0.17\ \Omega$ per pair round-trip), producing **$<0.2\text{ W}$** of total heat dissipation across the entire 0.4 m cord. **Rule of thumb:** Always keep ultra-thin 34 AWG cables strictly for short shelf/rack patches (0.2 m–1 m), and never bundle dozens of high-wattage thin PoE cables tightly inside an unventilated conduit!

### 2. Complete Physical Port Map & Cable Budget (Using Your 10-Pack)

```mermaid
flowchart TB
    subgraph Shelf ["Homelab Shelf Layout (Clustered within 0.4m / 1.3ft Reach)"]
        Switch["USW-Enterprise-8-PoE (10.0.0.203)<br>Keep Top Aluminum Heatsink Clear!"]
        Pi["Port 2 -> homelab-pi<br>10.0.0.200 (1 GbE)"]
        GPU["Port 3 -> homelab-gpu<br>10.0.0.201 (1 GbE / 2.5 GbE)"]
        Inj["Port 4 -> 60W PoE++ Injector<br>(LAN In -> PoE++ Out)"]
        UNAS["UniFi UNAS 2<br>10.0.0.202 (2.5 GbE)"]
        
        Switch ===|"Cable #1 (0.4m Blue)"| Pi
        Switch ===|"Cable #2 (0.4m Blue)"| GPU
        Switch ===|"Cable #3 (0.4m Blue - Data Only)"| Inj
        Inj ===|"Cable #4 (0.4m Blue - Data + 60W PoE++)"| UNAS
    end
    
    Router["Home Router (10.0.0.1)<br>(Port 1 Uplink)"] ===|"Cable #5 (0.4m Blue if adjacent,<br>or existing longer Cat 6 uplink)"| Switch
```

### 3. End-to-End Throughput & Bottleneck Analysis

Now that you have a **2.5 GbE** switch (`USW-Enterprise-8-PoE`) and a **2.5 GbE** NAS (`UNAS 2`), where are the real speed ceilings in your cluster?

| Data Path | Negotiated Link Speed | Realistic Max Throughput | True Bottleneck |
| :--- | :---: | :---: | :--- |
| **`homelab-pi` $\leftrightarrow$ `UNAS 2`** | 1 GbE (Pi) $\rightarrow$ 2.5 GbE (UNAS 2) | **~112–118 MB/s** (`~940 Mbps`) | Raspberry Pi 4's built-in 1 GbE NIC |
| **`homelab-gpu` (Built-in 1GbE) $\leftrightarrow$ `UNAS 2`** | 1 GbE $\rightarrow$ 2.5 GbE | **~112–118 MB/s** (`~940 Mbps`) | Laptop's built-in 1 GbE NIC |
| **`homelab-gpu` (w/ USB-C `RTL8156` 2.5GbE) $\leftrightarrow$ `UNAS 2`** | **2.5 GbE $\rightarrow$ 2.5 GbE** | **~170–220 MB/s** (`~1.4–1.8 Gbps`) | Sequential platter speed of 5400 RPM SATA HDDs (bursts to ~280 MB/s from UNAS 2's 4 GB RAM page cache!) |
| **`homelab-pi` + `homelab-gpu` Simultaneously $\rightarrow$ `UNAS 2`** | 1 GbE + 1 GbE $\rightarrow$ **2.5 GbE** | **~220+ MB/s aggregate** | **Zero switch congestion!** Because the UNAS 2 has a 2.5 GbE uplink, *both* 1 GbE nodes can hit the NAS at full speed simultaneously without saturating the NAS port! |

Look at that last row—even if both `homelab-pi` and `homelab-gpu` only have 1 GbE NICs today, the **2.5 GbE** port between `USW-Enterprise-8-PoE` and `UNAS 2` acts as a **multi-client aggregation pipe** so nightly backups from `homelab-pi` and workstation SMB transfers never starve each other!

## 📖 Official Hardware Reference Figure

<br>

<img src="https://github.com/rd908213/Homelab-Development/blob/main/assets/hardware/unifi_patch_cable_bl.png?raw=true" width="60%">

*Reference Figure 1: Ubiquiti UniFi Ultra-Thin Patch Cables (`UACC-CABLE-PATCH-0.4M-BL-10`, Blue variant center) — 0.4 m length, 2.5 mm TPE jacket, 34 AWG copper, ±90° vertical / ±45° horizontal bendable RJ45 boots, rated for 10 GbE and PoE+++.*

<br>

## 1.2.2 Step-by-Step Laboratory Guide

### Task 1.2.1: Structured Homelab Shelf Re-Cabling with `UACC-CABLE-PATCH-0.4M-BL-10`
1. Arrange your core hardware on your shelf/desk so every homelab node sits comfortably within the **0.4 m (1.3 ft)** reach of the `USW-Enterprise-8-PoE`:
   - Keep the **`USW-Enterprise-8-PoE`** central with at least **2–3 inches of clear open air** above its passive aluminum heatsink (never stack equipment on top of it).
   - Position the **`UNAS 2`** vertically next to the switch (giving its bottom intake and top exhaust vents unobstructed airflow) with its **60W PoE++ injector** tucked neatly behind/beside it on the shelf.
   - Keep `homelab-pi` and `homelab-gpu` (with its strain-relieved DC barrel cable from Chapter 2, Module 1.1!) adjacent to the switch.
2. Replace messy legacy cables with your **Blue `0.4m` UniFi Patch Cables**, gently bending the ±90° boots to route cleanly along the back edge of the shelf:
   - **Port 1:** Home Router Uplink *(use a `0.4m` blue cable if your router is right next to the switch, or keep your longer Cat 6 run if the router is across the room)*
   - **Port 2:** `homelab-pi` (`10.0.0.200`) — `0.4m` Blue Patch Cable
   - **Port 3:** `homelab-gpu` (`10.0.0.201`) — `0.4m` Blue Patch Cable
   - **Port 4:** `60W PoE++ Injector (LAN In)` — `0.4m` Blue Patch Cable
   - **Injector (`PoE Out`) $\rightarrow$ `UNAS 2`:** `0.4m` Blue Patch Cable
3. Store the remaining 5–6 unused `0.4m` blue patch cables in your homelab spares bin for future nodes (Ports 5–8).

### 🖼️ Task 1.2.1 Verification: Structured Homelab Shelf & `0.4m` Blue Patch Cabling

**Verification Objective:** Capture a clean photo of your finished homelab shelf layout showing the `USW-Enterprise-8-PoE`, `UNAS 2` + inline 60W PoE++ injector, `homelab-pi`, and `homelab-gpu` neatly interconnected with your blue `0.4m` UniFi patch cables.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 1.2.2: Reserve Static DHCP Leases (`10.0.0.202` & `10.0.0.203`) & Verify Link Negotiation
Now let's lock down our complete 4-device contiguous homelab IP table on your home router (`10.0.0.1`):

| Device | Hostname / Role | Reserved Static LAN IP |
| :--- | :--- | :---: |
| **Node 1** | `homelab-pi` (Raspberry Pi 4 — Immich Core, Pi-hole, Uptime Kuma) | `10.0.0.200` *(already reserved)* |
| **Node 2** | `homelab-gpu` (2019 ROG Zephyrus — CUDA ML, UniFi Network Server) | `10.0.0.201` *(already reserved)* |
| **Node 3** | `homelab-nas` (**UniFi UNAS 2** — RAID 1 Btrfs Storage) | **`10.0.0.202`** *(reserve now)* |
| **Core Switch** | `homelab-sw` (**UniFi `USW-Enterprise-8-PoE`** Management IP) | **`10.0.0.203`** *(reserve now)* |

1. Open your home router's administration portal (`http://10.0.0.1` or your gateway IP) and view the **Connected Devices / DHCP Client List**.
2. Locate the MAC addresses for:
   - Your **UniFi UNAS 2** (usually labeled `UNAS-2` or OUI vendor `Ubiquiti Inc.` — also printed on the underside label). Bind its MAC address to **`10.0.0.202`**.
   - Your **UniFi Switch Enterprise 8 PoE** (`USW-Enterprise-8-PoE` — MAC printed on underside label & visible on the 1.3" LCM info screen). Bind its MAC address to **`10.0.0.203`**.
3. Reboot (or briefly renew DHCP on) the `UNAS 2` and `USW-Enterprise-8-PoE` if they currently hold dynamic pool IPs, then verify all four core IPs answer sub-millisecond pings from `homelab-gpu` (`ssh homelab-gpu`):
   ```bash
   for ip in 10.0.0.200 10.0.0.201 10.0.0.202 10.0.0.203; do
       printf "Testing %-12s -> " "$ip"
       ping -c 2 -W 1 "$ip" | awk -F'/' '/rtt/ {print "OK (avg " $5 " ms)"}'
   done
   ```
   *(Expected output: All four homelab hosts respond `OK (avg ~0.2–0.5 ms)`!)*
4. Tap the **1.3" LCM touchscreen** on the `USW-Enterprise-8-PoE` and confirm **Port 4** shows a **2.5 GbE** negotiated link to the `UNAS 2` through the 60W PoE++ injector!

### 🖼️ Task 1.2.2 Verification: Contiguous 4-Host Ping Sweep & Router DHCP Reservations

**Verification Objective:** Capture a screenshot showing the ping sweep succeeding across `10.0.0.200`–`10.0.0.203` (and/or your router's DHCP Static Reservation table containing all four homelab devices).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Core shelf interconnects neatly wired using `UACC-CABLE-PATCH-0.4M-BL-10` blue patch cables with clean airflow clearance above `USW-Enterprise-8-PoE`.
- [ ] Static DHCP reservations bound for `UNAS 2` (`10.0.0.202`) and `USW-Enterprise-8-PoE` (`10.0.0.203`).
- [ ] All four homelab IP addresses (`10.0.0.200`–`10.0.0.203`) respond to sub-millisecond LAN pings.
- [ ] Switch LCM confirms **2.5 GbE** link negotiation on Port 4 (`UNAS 2` via 60W PoE++ injector).
- [ ] Verification screenshots/photos captured and attached above.

Proceed to [Module 2.1: Deploying the Self-Hosted UniFi Network Application on `homelab-gpu`](../02_unifi_network_management_and_switch_adoption/02_1_self_hosted_unifi_network_server.ipynb).